In [12]:
# load alice wonderful text data

with open("alice.txt","r",encoding="utf-8") as f:
  text=f.read()

start = text.find("CHAPTER I.")
text = text[start:]
print(text[:100])

CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race 


In [13]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])

In [15]:
# total unique words in the dataset
len(tokenizer.word_index)
# tokenizer.word_counts

3513

In [16]:
#split data into inp and output per sequence
input_seq=[]
max_len=0
for sentence in text.split("\n"):
  tokenize_sentence=tokenizer.texts_to_sequences([sentence])[0]
  max_len=max(max_len,len(tokenize_sentence))

  for i in range(1,len(tokenize_sentence)):
    input_seq.append(tokenize_sentence[:i+1])


In [17]:
# Add extra zeros to fulfil size according to max size sequence
padded_inp=pad_sequences(input_seq,maxlen=max_len,padding="pre")

In [18]:
X=padded_inp[:,:-1]
y=padded_inp[:,-1]


In [19]:
# make catagories from y(ouput)
from tensorflow.keras.utils import to_categorical
y=to_categorical(y,num_classes=3514)
y.shape

(27974, 3514)

In [20]:
from tensorflow.keras.layers import Dense,Embedding,LSTM
from tensorflow.keras.models import Sequential

In [82]:
#build archtecture of model
from tensorflow.keras.layers import Input

model = Sequential([
    Input(shape=(18,)),
    Embedding(input_dim=3514, output_dim=100),
    LSTM(128),
    Dense(3514, activation='softmax')
])

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 18, 100)        │       351,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3514)           │       453,306 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 921,954 (3.52 MB)

 Trainable params: 921,954 (3.52 MB)

 Non-trainable params: 0 (0.00 B)

In [83]:
# compile model
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

In [85]:
# train model
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    min_delta=0.001,
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.fit(X,y,epochs=100,callbacks=[early_stop])

Epoch 1/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1917 - loss: 4.5229
Epoch 2/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.2072 - loss: 4.2803
Epoch 3/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.2201 - loss: 4.0613
Epoch 4/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.2398 - loss: 3.8542
Epoch 5/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.2564 - loss: 3.6545
Epoch 6/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.2804 - loss: 3.4609
Epoch 7/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.3033 - loss: 3.2718
Epoch 8/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.3314 - loss: 3.0940
Epoch 9/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.3591 - loss: 2.9244
Epoch 10/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.3878 - loss: 2.7660
Epoch 11/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.4158 - loss: 2.6164
Epoch 12/100
875/875 ━━━━━━━━━━━━━━━━━━━

In [129]:
# predict next word
text="It"
for i in range(10):
  token_text=tokenizer.texts_to_sequences([text])[0]
  padded_token_text=pad_sequences([token_text],maxlen=18,padding="pre")
  pos=np.argmax(model.predict(padded_token_text,verbose=0))
  for word,index in tokenizer.word_index.items():
    if index==pos:
      text=text+" "+word
      print(text)





It was
It was the
It was the white
It was the white rabbit
It was the white rabbit trotting
It was the white rabbit trotting slowly
It was the white rabbit trotting slowly back
It was the white rabbit trotting slowly back again
It was the white rabbit trotting slowly back again and
It was the white rabbit trotting slowly back again and looking
